OVERVIEW – WHAT HAPPENED IN THIS NOTEBOOK

This notebook focused on completing the entire Exploratory Data Analysis (EDA)
and data preparation phase for a movie recommender system.

1. Raw movie metadata was inspected and understood conceptually, with a clear
   decision to build a similarity-based recommender (not a prediction model).

2. Text preprocessing was finalized:
   - Title and overview were combined into a semantic text field.
   - Safe truncation was applied to limit text length while preserving meaning.
   - Rollback safety was ensured by never overwriting raw columns.

3. Categorical features were reshaped:
   - Genres, keywords, and tagline were normalized (lowercased, cleaned).
   - These were merged into a structured semantic field for modeling.
   - The goal was to capture "what the movie is" separately from its plot.

4. A strict separation of concerns was enforced:
   - Similarity features → text_truncated, categorical_text
   - Filtering features → status
   - Ranking logic → production_companies, popularity, votes
   - Output metadata → runtime, title

5. Useless or noisy columns (budget, revenue, languages, images, URLs, etc.)
   were intentionally dropped to keep the dataset lightweight and model-focused.

6. A final clean dataframe (df_clean) was created containing only:
   - Identifiers
   - TF-IDF inputs
   - Filtering fields
   - Ranking metadata
   - Output display fields

7. The cleaned dataframe was saved as a CSV file (movies_clean.csv),
   marking the official end of EDA and preprocessing.

At this point, the dataset is frozen and ready for TF-IDF vectorization.
Any further changes are considered design decisions, not data cleaning.


In [ ]:
#download from -> https://www.kaggle.com/code/asaniczka/tmdb-movies-daily-updates/input

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('TMDB_movie6.csv')

In [ ]:
df.info()

In [ ]:
missing_count = df.isnull().sum()
missing_percent = (missing_count / len(df)) * 100

missing_df = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
}).sort_values(by="missing_percent", ascending=False)

missing_df


In [ ]:
df.duplicated().sum()

In [ ]:
df.replace("", pd.NA, inplace=True)

In [ ]:
df[df["title"].isnull() | df["original_title"].isnull()]

In [ ]:
df = df_original.copy(deep=True)

In [ ]:
df = df[df["title"].notnull()]

In [ ]:
df["title"].isnull().sum()

In [ ]:
text_cols = ["overview", "keywords", "tagline", "title"]

text_length_stats = {}

for col in text_cols:
    lengths = df[col].fillna("").str.len()
    text_length_stats[col] = {
        "min": lengths.min(),
        "median": lengths.median(),
        "max": lengths.max()
    }

pd.DataFrame(text_length_stats).T


In [ ]:
# Text columns used for content-based similarity
text_cols = ["overview", "keywords", "tagline", "title"]

# We measure text length to understand signal strength and sparsity.
# This step is diagnostic only — no cleaning or truncation here.
threshold = 10  # characters; below this is considered "near-empty"

for col in text_cols:
    # Convert NaN to empty string only for measurement safety
    lengths = df[col].fillna("").str.len()

    # Percentage of completely missing text
    empty_pct = (lengths == 0).mean() * 100

    # Percentage of text that is too short to be meaningful
    near_empty_pct = (lengths < threshold).mean() * 100

    print(
        f"{col}: "
        f"empty={empty_pct:.2f}%, "
        f"near-empty(<{threshold})={near_empty_pct:.2f}%"
    )

"""
INTERPRETATION NOTES (BASED ON OUTPUT):

overview:
- ~77% of movies have meaningful overview text
- Strong descriptive signal
- Should receive the highest weight in similarity
- Truncation may be needed due to long outliers (max ~1000 chars)

keywords:
- ~75% missing, but very informative when present
- Sparse but high-value signal
- Should be included with lower weight than overview

tagline:
- ~86% missing
- Weak and inconsistent signal
- Can be included as a very low-weight feature or ignored

title:
- Always present
- Short but critical identifier
- Needs cleaning due to long outliers (max ~324 chars)
- Should have moderate weight
"""


In [ ]:
df["overview"].fillna("").str.len().quantile(
    [0.90, 0.95, 0.98, 0.99, 0.995, 0.999]
)
#Quantiles reveal where the long-text cliff starts

#This avoids arbitrary cutoffs like “let’s use 500 chars”

In [ ]:
# Truncation decision for 'overview':
# - Based on quantile analysis
# - 99% of overviews are <= 900 characters
# - Truncating above 900 removes long-tail dominance
# - Affects ~1% of movies only

#OVERVIEW_MAX_LEN = 900


In [ ]:
df["title"].fillna("").str.len().quantile(
    [0.90, 0.95, 0.98, 0.99, 0.995, 0.999]
)

In [ ]:
# Truncation decision for 'title':
# - Titles are naturally short
# - >100 characters indicates corrupted or concatenated data
# - Truncate at 100 to prevent noise in similarity

#TITLE_MAX_LEN = 100


In [ ]:
df["status"].value_counts().head(10)


In [ ]:
df["runtime"].describe()


In [ ]:
df[df["title"].str.contains("Season", case=False, na=False)].shape
#titles with "season" written

## NOTE:
### Genre "TV Movie" represents made-for-television films, not TV series.
### These are valid single-movie entries and must be retained.
### TV series are filtered using runtime, not genre labels.


In [ ]:
movies_mask = df["runtime"].between(40, 300)

df[~movies_mask].shape


In [ ]:
df = df[movies_mask].copy()

In [ ]:
df["runtime"].describe()

In [ ]:
# Dataset scope decision:
# - This project is a movie-only recommender
# - TV series and episodic content are removed
# - Filtering is based on runtime (40–300 minutes)
# - Genre labels like "TV Movie" are retained
# - Resulting dataset ~570k movies


In [ ]:
df["title"].fillna("").str.len().quantile(
    [0.90, 0.95, 0.98, 0.99, 0.995, 0.999]
)
#TV titles often inflate length

#This check confirms whether truncation at 100 is still needed

In [ ]:
# -----------------------------
# STEP 1: SAFE TEXT TRUNCATION
# -----------------------------

# 1. Create backup columns for rollback safety
# These backups allow you to revert at any time
df["title_raw"] = df["title"]
df["overview_raw"] = df["overview"]

# 2. Handle missing values explicitly
# Missing text can break vectorizers later
df["title"] = df["title"].fillna("")
df["overview"] = df["overview"].fillna("")

# 3. Combine title + overview into a single text feature
# Title is repeated once to slightly increase its importance
df["text_full"] = df["title"] + ". " + df["overview"]

# 4. Define a safe truncation function
# This truncates by number of words, not characters (better semantic retention)
def truncate_text(text, max_words=120):
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words])

# 5. Apply truncation to create a NEW column
# Raw text is preserved in text_full
df["text_truncated"] = df["text_full"].apply(
    lambda x: truncate_text(x, max_words=120)
)

# 6. Optional: sanity check
# Compare lengths before and after truncation
df["text_len_before"] = df["text_full"].apply(lambda x: len(x.split()))
df["text_len_after"] = df["text_truncated"].apply(lambda x: len(x.split()))


In [ ]:
# ---------------------------------------
# STEP 2: CATEGORICAL FEATURE SHAPING
# ---------------------------------------

# 1. List categorical text columns to process
categorical_cols = ["genres", "keywords", "tagline"]

# 2. Fill missing values
# Empty string is safer than NaN for NLP pipelines
for col in categorical_cols:
    df[col] = df[col].fillna("")

# 3. Normalize separators and text
# - Lowercase
# - Replace separators with spaces
# - Remove extra whitespace
def normalize_categorical(text):
    text = text.lower()
    text = text.replace("|", " ")
    text = text.replace(",", " ")
    text = " ".join(text.split())
    return text

# 4. Apply normalization
for col in categorical_cols:
    df[col + "_clean"] = df[col].apply(normalize_categorical)

# 5. Combine all categorical features into one column
# This helps the model learn semantic co-occurrence
df["categorical_text"] = (
    df["genres_clean"] + " " +
    df["keywords_clean"] + " " +
    df["tagline_clean"]
)

# 6. Optional: inspect final shape
df[["categorical_text"]].head()


In [ ]:
# ---------------------------------------
# FINAL COLUMN SELECTION (STRICT)
# ---------------------------------------

# Columns required for the recommender system
columns_to_keep = [
    # Identifiers
    "id",
    "title",

    # TF-IDF inputs
    "text_truncated",
    "categorical_text",

    # Filtering
    "status",

    # Ranking / logic
    "production_companies",
    "vote_average",
    "vote_count",
    "popularity",

    # Output
    "runtime"
]

# Keep only columns that actually exist in the dataset
columns_to_keep = [col for col in columns_to_keep if col in df.columns]

# Create a clean working dataframe
df_clean = df[columns_to_keep].copy()

# Quick sanity check
df_clean.head()


In [ ]:
df_clean.shape

In [ ]:
# ---------------------------------------
# SAVE CLEAN DATAFRAME TO CSV
# ---------------------------------------

# Define output path / filename
output_path = "movies_clean.csv"

# Save dataframe
# index=False prevents pandas from adding an extra index column
df_clean.to_csv(output_path, index=False)

print(f"df_clean successfully saved to {output_path}")


In [ ]:
text_cols = ['genres', 'keywords', 'tagline', 'overview']
df[text_cols].head(3)


In [ ]:
df[text_cols] = df[text_cols].fillna('')

In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text)          # remove extra spaces
    return text.strip()


In [ ]:
for col in text_cols:
    df[col] = df[col].apply(clean_text)


In [ ]:
df[text_cols].sample(3)

In [ ]:
# 1. Verify text cleaning completed
print(df['overview'].sample(3))
print(df['keywords'].sample(3))

# 2. Check for completely empty movies
df['has_content'] = (
    (df['text_truncated'].str.len() > 10) | 
    (df['categorical_text'].str.len() > 10)
)
print(f"Movies with usable content: {df['has_content'].sum()}")

# 3. Inspect final distribution
print(df['text_truncated'].str.split().str.len().describe())
print(df['categorical_text'].str.split().str.len().describe())